03 · Data profiling

In [0]:
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
  SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS null_order_status,
  SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS null_order_purchase_timestamp,
  SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_order_approved_at,
  SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_order_delivered_carrier_date,
  SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_order_delivered_customer_date,
  SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS null_order_estimated_delivery_date

FROM olist.bronze.orders

In [0]:
-- orders: vacíos o solo espacios (bronze es todo string). Espera 0 en todas.
SELECT
  SUM(CASE WHEN TRIM(order_id) = '' THEN 1 ELSE 0 END) AS vacio_order_id,
  SUM(CASE WHEN TRIM(customer_id) = '' THEN 1 ELSE 0 END) AS vacio_customer_id,
  SUM(CASE WHEN TRIM(order_status) = '' THEN 1 ELSE 0 END) AS vacio_order_status,
  SUM(CASE WHEN TRIM(order_purchase_timestamp) = '' THEN 1 ELSE 0 END) AS vacio_purchase,
  SUM(CASE WHEN TRIM(order_approved_at) = '' THEN 1 ELSE 0 END) AS vacio_approved,
  SUM(CASE WHEN TRIM(order_delivered_carrier_date) = '' THEN 1 ELSE 0 END) AS vacio_carrier,
  SUM(CASE WHEN TRIM(order_delivered_customer_date) = '' THEN 1 ELSE 0 END) AS vacio_customer_date,
  SUM(CASE WHEN TRIM(order_estimated_delivery_date) = '' THEN 1 ELSE 0 END) AS vacio_estimated
FROM olist.bronze.orders;

In [0]:
SELECT
  MIN(TRY_CAST(order_purchase_timestamp AS TIMESTAMP)) AS purchase_timestamp_primera,
  MAX(TRY_CAST(order_purchase_timestamp AS TIMESTAMP)) AS purchase_timestamp_ultima,
  SUM(CASE WHEN order_purchase_timestamp IS NOT NULL
            AND TRY_CAST(order_purchase_timestamp AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS order_purchase_timestamp_invalido,
  MIN(TRY_CAST(order_approved_at AS TIMESTAMP)) AS order_approved_at_primera,
  MAX(TRY_CAST(order_approved_at AS TIMESTAMP)) AS order_approved_at_ultima,
  SUM(CASE WHEN order_approved_at IS NOT NULL
            AND TRY_CAST(order_approved_at AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS order_approved_at_invalido,
  MIN(TRY_CAST(order_delivered_carrier_date AS TIMESTAMP)) AS order_delivered_carrier_date_primera,
  MAX(TRY_CAST(order_delivered_carrier_date AS TIMESTAMP)) AS order_delivered_carrier_date_ultima,
  SUM(CASE WHEN order_delivered_carrier_date IS NOT NULL
            AND TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS order_delivered_carrier_date_invalido,
  MIN(TRY_CAST(order_delivered_customer_date AS TIMESTAMP)) AS order_delivered_customer_date_primera,
  MAX(TRY_CAST(order_delivered_customer_date AS TIMESTAMP)) AS order_delivered_customer_date_ultima,
  SUM(CASE WHEN order_delivered_customer_date IS NOT NULL
            AND TRY_CAST(order_delivered_customer_date AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS order_delivered_customer_date_invalido,
  MIN(TRY_CAST(order_estimated_delivery_date AS TIMESTAMP)) AS order_estimated_delivery_date_primera,
  MAX(TRY_CAST(order_estimated_delivery_date AS TIMESTAMP)) AS order_estimated_delivery_date_ultima,
  SUM(CASE WHEN order_estimated_delivery_date IS NOT NULL
            AND TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS order_estimated_delivery_date_invalido

FROM olist.bronze.orders;

In [0]:
SELECT COUNT(*) AS con_hora
FROM olist.bronze.orders
WHERE TRY_CAST(order_estimated_delivery_date AS TIMESTAMP)
      <> DATE_TRUNC('DAY', TRY_CAST(order_estimated_delivery_date AS TIMESTAMP));

In [0]:
SELECT COUNT(*) AS imposibles
FROM olist.bronze.orders
WHERE TRY_CAST(order_delivered_customer_date AS TIMESTAMP)
      < TRY_CAST(order_purchase_timestamp AS TIMESTAMP);

In [0]:
WITH numeradas AS (
  SELECT
    order_id,
    ROW_NUMBER() OVER (
      PARTITION BY order_id
      ORDER BY order_purchase_timestamp DESC, customer_id DESC
    ) AS rn
  FROM olist.bronze.orders
)
SELECT COUNT(*) AS filas_sobrantes
FROM numeradas
WHERE rn > 1;

In [0]:
SELECT
  order_status,
  COUNT(*) AS total_ordenes,
  COUNT(*) - COUNT(order_approved_at) AS sin_aprobacion,
  COUNT(*) - COUNT(order_delivered_carrier_date) AS sin_envio_transportista,
  COUNT(*) - COUNT(order_delivered_customer_date) AS sin_entrega_cliente
FROM olist.bronze.orders
GROUP BY order_status
ORDER BY total_ordenes DESC;

## orders (99 441 filas)
- Nulos solo en fechas del ciclo de vida: approved_at 160, delivered_carrier_date 1 783, delivered_customer_date 2 965. Forman un embudo: los nulos son pedidos que aún no llegan a esa etapa. 0 vacíos en las 8 columnas.
- order_id único (0 filas sobrantes con ROW_NUMBER).
- Fechas sin inválidos: compra del 2016-09-04 al 2018-10-17; entrega estimada del 2016-09-30 al 2018-11-12, siempre a las 00:00 (0 con hora). 0 entregas antes de la compra.
- Estados: delivered 96 478, shipped 1 107, canceled 625, unavailable 609, invoiced 314, processing 301, created 5, approved 2.
- Inconsistencias pequeñas: 8 pedidos delivered sin fecha de entrega y 6 canceled con fecha de entrega.
- Decisión para silver: no rellenar nulos; la entrega tardía se compara por fecha (DATE), porque la estimada no tiene hora.

In [0]:
SELECT
  COUNT(*) AS total_filas,
  COUNT(*) - COUNT(customer_id) AS null_customer_id,
  COUNT(*) - COUNT(customer_unique_id) AS null_customer_unique_id,
  COUNT(*) - COUNT(customer_zip_code_prefix) AS null_zip,
  COUNT(*) - COUNT(customer_city) AS null_city,
  COUNT(*) - COUNT(customer_state) AS null_state,
  SUM(CASE WHEN TRIM(customer_id) = '' THEN 1 ELSE 0 END) AS vacio_customer_id,
  SUM(CASE WHEN TRIM(customer_unique_id) = '' THEN 1 ELSE 0 END) AS vacio_customer_unique_id,
  SUM(CASE WHEN TRIM(customer_zip_code_prefix) = '' THEN 1 ELSE 0 END) AS vacio_zip,
  SUM(CASE WHEN TRIM(customer_city) = '' THEN 1 ELSE 0 END) AS vacio_city,
  SUM(CASE WHEN TRIM(customer_state) = '' THEN 1 ELSE 0 END) AS vacio_state
FROM olist.bronze.customers;

In [0]:
WITH duplicados_customer AS (
  SELECT customer_id, COUNT(*) AS n
  FROM olist.bronze.customers
  GROUP BY customer_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM duplicados_customer;

In [0]:
WITH por_persona AS (
  SELECT customer_unique_id, COUNT(*) AS n
  FROM olist.bronze.customers
  GROUP BY customer_unique_id
)
SELECT
  COUNT(*) AS personas_unicas,
  SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS personas_con_varias_compras
FROM por_persona;

In [0]:
-- Espacios sobrantes antes/después en customer_state
SELECT
  SUM(CASE WHEN customer_state <> TRIM(customer_state) THEN 1 ELSE 0 END) AS con_espacios_state,
  SUM(CASE WHEN LENGTH(customer_state) <> 2 THEN 1 ELSE 0 END) AS largo_crudo_distinto_2
FROM olist.bronze.customers;



In [0]:
-- Distribución de longitudes en customer_zip_code_prefix
SELECT
  LENGTH(customer_zip_code_prefix) AS largo_zip,
  COUNT(*) AS filas
FROM olist.bronze.customers
GROUP BY LENGTH(customer_zip_code_prefix)
ORDER BY largo_zip;

In [0]:
SELECT
  COUNT(DISTINCT customer_city) AS ciudades_crudas,
  COUNT(DISTINCT LOWER(TRIM(customer_city))) AS ciudades_normalizadas
FROM olist.bronze.customers;

In [0]:
-- customer_city con caracteres raros o espacios sobrantes.
SELECT
  COUNT(*) AS filas_sucias,
  COUNT(DISTINCT customer_city) AS ciudades_sucias
FROM olist.bronze.customers
WHERE customer_city RLIKE '[^a-zA-Z0-9 -]'
   OR customer_city <> TRIM(customer_city);

In [0]:
-- Listado de customer_city sucias, para separar suciedad real de apóstrofes válidos.
SELECT customer_city, COUNT(*) AS filas
FROM olist.bronze.customers
WHERE customer_city RLIKE '[^a-zA-Z0-9 -]'
   OR customer_city <> TRIM(customer_city)
GROUP BY customer_city
ORDER BY filas DESC;

In [0]:

SELECT COUNT(*) AS orders_sin_customer
FROM olist.bronze.orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.customers c
  WHERE c.customer_id = o.customer_id
);

In [0]:

SELECT COUNT(*) AS customers_sin_pedidos
FROM olist.bronze.customers c
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.orders o
  WHERE o.customer_id = c.customer_id
);




## customers (99 441 filas)
- 0 nulos y 0 vacíos en las 5 columnas. customer_id único y los 99 441 pedidos tienen cliente (y viceversa).
- customer_id es uno por pedido; la persona real es customer_unique_id: 96 096 personas, 2 997 con más de una compra.
- Estado: 2 caracteres y sin espacios. Código postal: las 99 441 filas miden 5.
- Ciudades: 4 119 (crudas = normalizadas). El RLIKE marcó 226 filas en 32 ciudades, pero son apóstrofes de nombres reales ("santa barbara d'oeste", "dias d'avila"): falsos positivos, no suciedad.
- Decisión para silver: dim_customer por customer_unique_id; ciudad solo con lower y trim.

In [0]:
DESCRIBE olist.bronze.order_items;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.order_items;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  -- Nulos
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(order_item_id) AS null_order_item_id,
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(seller_id) AS null_seller_id,
  COUNT(*) - COUNT(shipping_limit_date) AS null_shipping_limit_date,
  COUNT(*) - COUNT(price) AS null_price,
  COUNT(*) - COUNT(freight_value) AS null_freight_value,
  -- Vacíos o solo espacios
  SUM(CASE WHEN TRIM(order_id) = '' THEN 1 ELSE 0 END) AS vacio_order_id,
  SUM(CASE WHEN TRIM(order_item_id) = '' THEN 1 ELSE 0 END) AS vacio_order_item_id,
  SUM(CASE WHEN TRIM(product_id) = '' THEN 1 ELSE 0 END) AS vacio_product_id,
  SUM(CASE WHEN TRIM(seller_id) = '' THEN 1 ELSE 0 END) AS vacio_seller_id,
  SUM(CASE WHEN TRIM(shipping_limit_date) = '' THEN 1 ELSE 0 END) AS vacio_shipping_limit_date,
  SUM(CASE WHEN TRIM(price) = '' THEN 1 ELSE 0 END) AS vacio_price,
  SUM(CASE WHEN TRIM(freight_value) = '' THEN 1 ELSE 0 END) AS vacio_freight_value
FROM olist.bronze.order_items;

In [0]:
WITH dup AS (
  SELECT
    order_id,
    order_item_id,
    COUNT(*) AS n
  FROM olist.bronze.order_items
  GROUP BY order_id, order_item_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM dup;

In [0]:
SELECT
  -- price
  MIN(TRY_CAST(price AS DECIMAL(10,2))) AS price_min,
  MAX(TRY_CAST(price AS DECIMAL(10,2))) AS price_max,
  SUM(CASE WHEN price IS NOT NULL
           AND TRY_CAST(price AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS price_invalido,
  SUM(CASE WHEN TRY_CAST(price AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS price_no_positivo,
  -- freight_value
  MIN(TRY_CAST(freight_value AS DECIMAL(10,2))) AS freight_min,
  MAX(TRY_CAST(freight_value AS DECIMAL(10,2))) AS freight_max,
  SUM(CASE WHEN freight_value IS NOT NULL
           AND TRY_CAST(freight_value AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS freight_invalido,
  SUM(CASE WHEN TRY_CAST(freight_value AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS freight_no_positivo
FROM olist.bronze.order_items;

In [0]:
SELECT
  MIN(TRY_CAST(shipping_limit_date AS TIMESTAMP)) AS shipping_limit_min,
  MAX(TRY_CAST(shipping_limit_date AS TIMESTAMP)) AS shipping_limit_max,
  SUM(CASE WHEN shipping_limit_date IS NOT NULL
            AND TRY_CAST(shipping_limit_date AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS shipping_limit_invalido
FROM olist.bronze.order_items;

In [0]:
SELECT COUNT(*) AS items_sin_pedido
FROM olist.bronze.order_items oi
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.orders o
  WHERE o.order_id = oi.order_id
);

In [0]:
SELECT COUNT(*) AS items_sin_producto
FROM olist.bronze.order_items oi
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.products p
  WHERE p.product_id = oi.product_id
);

In [0]:
SELECT COUNT(*) AS items_sin_vendedor
FROM olist.bronze.order_items oi
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.sellers s
  WHERE s.seller_id = oi.seller_id
);

In [0]:
SELECT COUNT(*) AS pedidos_sin_items
FROM olist.bronze.orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_items oi
  WHERE oi.order_id = o.order_id
);

In [0]:
SELECT COUNT(*) AS items_post_2018
FROM olist.bronze.order_items
WHERE TRY_CAST(shipping_limit_date AS TIMESTAMP) > TIMESTAMP '2018-12-31 23:59:59';

In [0]:
SELECT
  order_id,
  order_item_id,
  product_id,
  seller_id,
  TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_ts
FROM olist.bronze.order_items
WHERE shipping_limit_date IS NOT NULL
ORDER BY shipping_limit_ts DESC
LIMIT 5;

In [0]:
SELECT
  o.order_status,
  COUNT(*) AS pedidos_sin_items
FROM olist.bronze.orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_items oi
  WHERE oi.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY pedidos_sin_items DESC;

## order_items (112 650 filas)
- 0 nulos y 0 vacíos. Llave compuesta (order_id, order_item_id) sin repetidos.
- price de 0.85 a 6 735.00, sin inválidos ni ceros. freight_value de 0.00 a 409.68: 383 ítems con flete 0 (envío gratis, plausible).
- shipping_limit_date del 2016-09-19 al 2020-04-09: 4 ítems después de 2018 (3 de ellos del mismo vendedor), fuera del rango del dataset.
- Integridad: 0 ítems sin pedido, sin producto o sin vendedor. 775 pedidos sin ítems: unavailable 603, canceled 164, created 5, invoiced 2, shipped 1.
- Decisión para silver: conservar los 4 ítems de 2020 y convertirlos en alerta en P1-12; no se borran.

In [0]:
DESCRIBE olist.bronze.products;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.products;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  -- Nulos
  COUNT(*) - COUNT(product_id) AS null_product_id,
  COUNT(*) - COUNT(product_category_name) AS null_category,
  COUNT(*) - COUNT(product_name_lenght) AS null_name_len,
  COUNT(*) - COUNT(product_description_lenght) AS null_desc_len,
  COUNT(*) - COUNT(product_photos_qty) AS null_photos_qty,
  COUNT(*) - COUNT(product_weight_g) AS null_weight_g,
  COUNT(*) - COUNT(product_length_cm) AS null_length_cm,
  COUNT(*) - COUNT(product_height_cm) AS null_height_cm,
  COUNT(*) - COUNT(product_width_cm) AS null_width_cm,
  -- Vacíos o solo espacios
  SUM(CASE WHEN TRIM(product_id) = '' THEN 1 ELSE 0 END) AS vacio_product_id,
  SUM(CASE WHEN TRIM(product_category_name) = '' THEN 1 ELSE 0 END) AS vacio_category,
  SUM(CASE WHEN TRIM(product_name_lenght) = '' THEN 1 ELSE 0 END) AS vacio_name_len,
  SUM(CASE WHEN TRIM(product_description_lenght) = '' THEN 1 ELSE 0 END) AS vacio_desc_len,
  SUM(CASE WHEN TRIM(product_photos_qty) = '' THEN 1 ELSE 0 END) AS vacio_photos_qty,
  SUM(CASE WHEN TRIM(product_weight_g) = '' THEN 1 ELSE 0 END) AS vacio_weight_g,
  SUM(CASE WHEN TRIM(product_length_cm) = '' THEN 1 ELSE 0 END) AS vacio_length_cm,
  SUM(CASE WHEN TRIM(product_height_cm) = '' THEN 1 ELSE 0 END) AS vacio_height_cm,
  SUM(CASE WHEN TRIM(product_width_cm) = '' THEN 1 ELSE 0 END) AS vacio_width_cm
FROM olist.bronze.products;

In [0]:
WITH dup AS (
  SELECT
    product_id,
    COUNT(*) AS n
  FROM olist.bronze.products
  GROUP BY product_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM dup;

In [0]:
SELECT
  -- Largo del nombre
  MIN(TRY_CAST(product_name_lenght AS DECIMAL(10,2))) AS name_len_min,
  MAX(TRY_CAST(product_name_lenght AS DECIMAL(10,2))) AS name_len_max,
  SUM(CASE WHEN product_name_lenght IS NOT NULL
            AND TRY_CAST(product_name_lenght AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS name_len_invalido,
  SUM(CASE WHEN TRY_CAST(product_name_lenght AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS name_len_no_positivo,
  -- Largo de la descripción
  MIN(TRY_CAST(product_description_lenght AS DECIMAL(10,2))) AS desc_len_min,
  MAX(TRY_CAST(product_description_lenght AS DECIMAL(10,2))) AS desc_len_max,
  SUM(CASE WHEN product_description_lenght IS NOT NULL
            AND TRY_CAST(product_description_lenght AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS desc_len_invalido,
  SUM(CASE WHEN TRY_CAST(product_description_lenght AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS desc_len_no_positivo,
  -- Cantidad de fotos
  MIN(TRY_CAST(product_photos_qty AS DECIMAL(10,2))) AS photos_qty_min,
  MAX(TRY_CAST(product_photos_qty AS DECIMAL(10,2))) AS photos_qty_max,
  SUM(CASE WHEN product_photos_qty IS NOT NULL
            AND TRY_CAST(product_photos_qty AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS photos_qty_invalido,
  SUM(CASE WHEN TRY_CAST(product_photos_qty AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS photos_qty_no_positivo,
  -- Peso en gramos
  MIN(TRY_CAST(product_weight_g AS DECIMAL(10,2))) AS weight_g_min,
  MAX(TRY_CAST(product_weight_g AS DECIMAL(10,2))) AS weight_g_max,
  SUM(CASE WHEN product_weight_g IS NOT NULL
            AND TRY_CAST(product_weight_g AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS weight_g_invalido,
  SUM(CASE WHEN TRY_CAST(product_weight_g AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS weight_g_no_positivo,
  -- Largo en cm
  MIN(TRY_CAST(product_length_cm AS DECIMAL(10,2))) AS length_cm_min,
  MAX(TRY_CAST(product_length_cm AS DECIMAL(10,2))) AS length_cm_max,
  SUM(CASE WHEN product_length_cm IS NOT NULL
            AND TRY_CAST(product_length_cm AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS length_cm_invalido,
  SUM(CASE WHEN TRY_CAST(product_length_cm AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS length_cm_no_positivo,
  -- Alto en cm
  MIN(TRY_CAST(product_height_cm AS DECIMAL(10,2))) AS height_cm_min,
  MAX(TRY_CAST(product_height_cm AS DECIMAL(10,2))) AS height_cm_max,
  SUM(CASE WHEN product_height_cm IS NOT NULL
            AND TRY_CAST(product_height_cm AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS height_cm_invalido,
  SUM(CASE WHEN TRY_CAST(product_height_cm AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS height_cm_no_positivo,
  -- Ancho en cm
  MIN(TRY_CAST(product_width_cm AS DECIMAL(10,2))) AS width_cm_min,
  MAX(TRY_CAST(product_width_cm AS DECIMAL(10,2))) AS width_cm_max,
  SUM(CASE WHEN product_width_cm IS NOT NULL
            AND TRY_CAST(product_width_cm AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS width_cm_invalido,
  SUM(CASE WHEN TRY_CAST(product_width_cm AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS width_cm_no_positivo
FROM olist.bronze.products;

In [0]:
SELECT
  SUM(
    CASE
      WHEN product_category_name IS NULL
       AND product_name_lenght IS NULL
       AND product_description_lenght IS NULL
       AND product_photos_qty IS NULL
      THEN 1 ELSE 0
    END
  ) AS cuatro_campos_nulos
FROM olist.bronze.products;

In [0]:
SELECT COUNT(DISTINCT product_category_name) AS categorias_distintas
FROM olist.bronze.products
WHERE product_category_name IS NOT NULL;

In [0]:
SELECT
  COUNT(*) AS productos_sin_traduccion,
  COUNT(DISTINCT p.product_category_name) AS categorias_sin_traduccion
FROM olist.bronze.products p
WHERE p.product_category_name IS NOT NULL
  AND NOT EXISTS (
    SELECT 1
    FROM olist.bronze.category_translation t
    WHERE t.product_category_name = p.product_category_name
  );

In [0]:
-- Categorías de products sin traducción al inglés. Espera 2 filas.
SELECT
  p.product_category_name,
  COUNT(*) AS productos
FROM olist.bronze.products p
WHERE p.product_category_name IS NOT NULL
  AND NOT EXISTS (
    SELECT 1
    FROM olist.bronze.category_translation t
    WHERE t.product_category_name = p.product_category_name
  )
GROUP BY p.product_category_name
ORDER BY productos DESC;

In [0]:
SELECT COUNT(*) AS productos_sin_items
FROM olist.bronze.products p
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_items oi
  WHERE oi.product_id = p.product_id
);

## products (32 951 filas)
- product_id único. 610 productos con 4 campos nulos a la vez (categoría, largo del nombre, largo de la descripción y fotos); 2 sin medidas ni peso.
- Rangos sin inválidos; 4 productos con peso 0.
- 73 categorías: 71 traducidas y 2 sin traducción (portateis_cozinha_e_preparadores_de_alimentos con 10 productos y pc_gamer con 3).
- Todos los productos aparecen en al menos un ítem.
- Decisiones para silver: renombrar lenght → length; peso 0 → NULL; LEFT JOIN con la traducción + COALESCE con el nombre original; sin categoría → 'unknown'.

In [0]:
DESCRIBE olist.bronze.sellers;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.sellers;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  COUNT(*) - COUNT(seller_id) AS null_seller_id,
  COUNT(*) - COUNT(seller_zip_code_prefix) AS null_zip,
  COUNT(*) - COUNT(seller_city) AS null_city,
  COUNT(*) - COUNT(seller_state) AS null_state,
  SUM(CASE WHEN TRIM(seller_id) = '' THEN 1 ELSE 0 END) AS vacio_seller_id,
  SUM(CASE WHEN TRIM(seller_zip_code_prefix) = '' THEN 1 ELSE 0 END) AS vacio_zip,
  SUM(CASE WHEN TRIM(seller_city) = '' THEN 1 ELSE 0 END) AS vacio_city,
  SUM(CASE WHEN TRIM(seller_state) = '' THEN 1 ELSE 0 END) AS vacio_state
FROM olist.bronze.sellers;

In [0]:
WITH dup AS (
  SELECT seller_id, COUNT(*) AS n
  FROM olist.bronze.sellers
  GROUP BY seller_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM dup;

In [0]:
SELECT
  COUNT(DISTINCT seller_state) AS estados_distintos,
  SUM(CASE WHEN LENGTH(seller_state) <> 2 THEN 1 ELSE 0 END) AS largo_crudo_distinto_2,
  SUM(CASE WHEN seller_state <> TRIM(seller_state) THEN 1 ELSE 0 END) AS con_espacios_state
FROM olist.bronze.sellers;

In [0]:
SELECT
  COUNT(DISTINCT seller_city) AS ciudades_crudas,
  COUNT(DISTINCT LOWER(TRIM(seller_city))) AS ciudades_normalizadas
FROM olist.bronze.sellers;

In [0]:
SELECT seller_city, COUNT(*) AS filas
FROM olist.bronze.sellers
WHERE seller_city RLIKE '[^a-zA-Z0-9 -]'
   OR seller_city <> TRIM(seller_city)
GROUP BY seller_city
ORDER BY filas DESC
LIMIT 20;

In [0]:
-- Cuántas filas y ciudades de sellers tienen texto sucio.
SELECT
  COUNT(*) AS filas_sucias,
  COUNT(DISTINCT seller_city) AS ciudades_sucias
FROM olist.bronze.sellers
WHERE seller_city RLIKE '[^a-zA-Z0-9 -]'
   OR seller_city <> TRIM(seller_city);

In [0]:
SELECT
  LENGTH(seller_zip_code_prefix) AS largo_zip,
  COUNT(*) AS filas
FROM olist.bronze.sellers
GROUP BY LENGTH(seller_zip_code_prefix)
ORDER BY largo_zip;

In [0]:
SELECT COUNT(*) AS vendedores_sin_items
FROM olist.bronze.sellers s
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_items oi
  WHERE oi.seller_id = s.seller_id
);

## sellers (3 095 filas)
- 0 nulos y 0 vacíos. seller_id único y todos los vendedores tienen ítems.
- Estados: 23, de 2 caracteres y sin espacios. Código postal: las 3 095 filas miden 5.
- Ciudades: 611 (crudas = normalizadas), pero 28 filas en 24 ciudades con texto sucio: estado o ciudad pegada ("sp / sp", "pinhais/pr"), texto entre paréntesis y dos apóstrofes distintos para la misma ciudad ("santa barbara d'oeste" y "d´oeste").
- Decisión para silver: seller_city solo con lower y trim (no se usa en las preguntas de negocio); la geografía se analiza con seller_state.

In [0]:
DESCRIBE olist.bronze.order_payments;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.order_payments;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(payment_sequential) AS null_payment_sequential,
  COUNT(*) - COUNT(payment_type) AS null_payment_type,
  COUNT(*) - COUNT(payment_installments) AS null_payment_installments,
  COUNT(*) - COUNT(payment_value) AS null_payment_value,
  SUM(CASE WHEN TRIM(CAST(order_id AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_order_id,
  SUM(CASE WHEN TRIM(CAST(payment_sequential AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_payment_sequential,
  SUM(CASE WHEN TRIM(CAST(payment_type AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_payment_type,
  SUM(CASE WHEN TRIM(CAST(payment_installments AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_payment_installments,
  SUM(CASE WHEN TRIM(CAST(payment_value AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_payment_value
FROM olist.bronze.order_payments;

In [0]:
WITH dup AS (
  SELECT order_id, payment_sequential, COUNT(*) AS n
  FROM olist.bronze.order_payments
  GROUP BY order_id, payment_sequential
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM dup;

In [0]:
SELECT
  payment_type,
  COUNT(*) AS filas
FROM olist.bronze.order_payments
GROUP BY payment_type
ORDER BY filas DESC;

In [0]:
SELECT
  -- payment_sequential
  MIN(TRY_CAST(payment_sequential AS DECIMAL(10,2))) AS seq_min,
  MAX(TRY_CAST(payment_sequential AS DECIMAL(10,2))) AS seq_max,
  SUM(CASE WHEN payment_sequential IS NOT NULL
            AND TRY_CAST(payment_sequential AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS seq_invalido,
  SUM(CASE WHEN TRY_CAST(payment_sequential AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS seq_no_positivo,
  -- payment_installments
  MIN(TRY_CAST(payment_installments AS DECIMAL(10,2))) AS inst_min,
  MAX(TRY_CAST(payment_installments AS DECIMAL(10,2))) AS inst_max,
  SUM(CASE WHEN payment_installments IS NOT NULL
            AND TRY_CAST(payment_installments AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS inst_invalido,
  SUM(CASE WHEN TRY_CAST(payment_installments AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS inst_no_positivo,
  -- payment_value
  MIN(TRY_CAST(payment_value AS DECIMAL(10,2))) AS val_min,
  MAX(TRY_CAST(payment_value AS DECIMAL(10,2))) AS val_max,
  SUM(CASE WHEN payment_value IS NOT NULL
            AND TRY_CAST(payment_value AS DECIMAL(10,2)) IS NULL
           THEN 1 ELSE 0 END) AS val_invalido,
  SUM(CASE WHEN TRY_CAST(payment_value AS DECIMAL(10,2)) <= 0 THEN 1 ELSE 0 END) AS val_no_positivo
FROM olist.bronze.order_payments;

In [0]:
SELECT COUNT(*) AS pagos_sin_pedido
FROM olist.bronze.order_payments op
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.orders o
  WHERE o.order_id = op.order_id
);

In [0]:
SELECT COUNT(*) AS pedidos_sin_pago
FROM olist.bronze.orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_payments op
  WHERE op.order_id = o.order_id
);

In [0]:
-- Pagos con valor 0: ¿qué tipo de pago son? Espera 9 filas.
SELECT order_id, payment_sequential, payment_type, payment_installments, payment_value
FROM olist.bronze.order_payments
WHERE TRY_CAST(payment_value AS DECIMAL(10,2)) <= 0
ORDER BY payment_type, order_id;

In [0]:
-- Pagos con 0 cuotas. Espera 2 filas.
SELECT order_id, payment_sequential, payment_type, payment_installments, payment_value
FROM olist.bronze.order_payments
WHERE TRY_CAST(payment_installments AS INT) <= 0;

In [0]:
-- Pedido sin pago y su estado. Espera 1 fila.
SELECT o.order_id, o.order_status
FROM olist.bronze.orders o
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.order_payments op
  WHERE op.order_id = o.order_id
);

In [0]:
-- Pedidos cuya secuencia de pagos tiene huecos (máximo ≠ cantidad de filas). Espera 0 o pocos.
SELECT COUNT(*) AS pedidos_con_huecos
FROM (
  SELECT order_id
  FROM olist.bronze.order_payments
  GROUP BY order_id
  HAVING MAX(CAST(payment_sequential AS INT)) <> COUNT(*)
);

## order_payments (103 886 filas)
- 0 nulos y 0 vacíos. Llave compuesta (order_id, payment_sequential) sin repetidos.
- payment_type: credit_card 76 795, boleto 19 784, voucher 5 775, debit_card 1 529, not_defined 3.
- payment_sequential de 1 a 29; 80 pedidos con huecos en la secuencia (falta algún número intermedio).
- payment_value de 0.00 a 13 664.08. Hay 9 pagos en 0: los 3 not_defined y 6 vouchers dentro de pedidos con varios pagos. No alteran el total por pedido.
- payment_installments de 0 a 24: 2 pagos credit_card con 0 cuotas (inconsistente: el mínimo es 1).
- Integridad: 0 pagos sin pedido; 1 pedido delivered sin pago (anomalía real).
- Decisiones para silver: sequential e installments a INT y value a DECIMAL(10,2); no imputar ni renumerar; pago por pedido = SUM(payment_value) por order_id; el pedido sin pago queda con pago NULL en fact_orders. Las cuotas en 0 y el pedido sin pago pasan a alertas en P1-12.

In [0]:
DESCRIBE olist.bronze.order_reviews;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.order_reviews;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  COUNT(*) - COUNT(review_id) AS null_review_id,
  COUNT(*) - COUNT(order_id) AS null_order_id,
  COUNT(*) - COUNT(review_score) AS null_review_score,
  COUNT(*) - COUNT(review_comment_title) AS null_title,
  COUNT(*) - COUNT(review_comment_message) AS null_message,
  COUNT(*) - COUNT(review_creation_date) AS null_creation_date,
  COUNT(*) - COUNT(review_answer_timestamp) AS null_answer_ts,
  SUM(CASE WHEN TRIM(CAST(review_id AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_review_id,
  SUM(CASE WHEN TRIM(CAST(order_id AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_order_id,
  SUM(CASE WHEN TRIM(CAST(review_score AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_review_score,
  SUM(CASE WHEN TRIM(CAST(review_comment_title AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_title,
  SUM(CASE WHEN TRIM(CAST(review_comment_message AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_message,
  SUM(CASE WHEN TRIM(CAST(review_creation_date AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_creation_date,
  SUM(CASE WHEN TRIM(CAST(review_answer_timestamp AS STRING)) = '' THEN 1 ELSE 0 END) AS vacio_answer_ts
FROM olist.bronze.order_reviews;

In [0]:
WITH dup_review AS (
  SELECT review_id, COUNT(*) AS n
  FROM olist.bronze.order_reviews
  GROUP BY review_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS review_ids_repetidos
FROM dup_review;

In [0]:
WITH por_pedido AS (
  SELECT order_id, COUNT(*) AS n
  FROM olist.bronze.order_reviews
  GROUP BY order_id
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS pedidos_con_varias_resenas
FROM por_pedido;

In [0]:
SELECT
  review_score,
  COUNT(*) AS filas
FROM olist.bronze.order_reviews
GROUP BY review_score
ORDER BY review_score;

In [0]:
SELECT
  MIN(TRY_CAST(review_creation_date AS TIMESTAMP)) AS creation_min,
  MAX(TRY_CAST(review_creation_date AS TIMESTAMP)) AS creation_max,
  SUM(CASE WHEN review_creation_date IS NOT NULL
            AND TRY_CAST(review_creation_date AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS creation_invalido,
  MIN(TRY_CAST(review_answer_timestamp AS TIMESTAMP)) AS answer_min,
  MAX(TRY_CAST(review_answer_timestamp AS TIMESTAMP)) AS answer_max,
  SUM(CASE WHEN review_answer_timestamp IS NOT NULL
            AND TRY_CAST(review_answer_timestamp AS TIMESTAMP) IS NULL
           THEN 1 ELSE 0 END) AS answer_invalido,
  SUM(CASE WHEN TRY_CAST(review_answer_timestamp AS TIMESTAMP)
              < TRY_CAST(review_creation_date AS TIMESTAMP)
           THEN 1 ELSE 0 END) AS respuesta_antes_de_creacion
FROM olist.bronze.order_reviews;

In [0]:
SELECT COUNT(*) AS resenas_sin_pedido
FROM olist.bronze.order_reviews r
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.orders o
  WHERE o.order_id = r.order_id
);

In [0]:
DESCRIBE olist.bronze.category_translation;

In [0]:
SELECT COUNT(*) AS total_filas
FROM olist.bronze.category_translation;

In [0]:
SELECT
  COUNT(*) AS total_filas,
  COUNT(*) - COUNT(product_category_name) AS null_pt_category,
  COUNT(*) - COUNT(product_category_name_english) AS null_en_category,
  SUM(CASE WHEN TRIM(product_category_name) = '' THEN 1 ELSE 0 END) AS vacio_pt_category,
  SUM(CASE WHEN TRIM(product_category_name_english) = '' THEN 1 ELSE 0 END) AS vacio_en_category
FROM olist.bronze.category_translation;

In [0]:
WITH dup AS (
  SELECT product_category_name, COUNT(*) AS n
  FROM olist.bronze.category_translation
  GROUP BY product_category_name
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS llaves_repetidas
FROM dup;

In [0]:
-- ¿Dos categorías en portugués comparten el mismo nombre en inglés? Espera 0.
SELECT COUNT(*) AS nombres_en_repetidos
FROM (
  SELECT product_category_name_english
  FROM olist.bronze.category_translation
  GROUP BY product_category_name_english
  HAVING COUNT(*) > 1
);

In [0]:
SELECT COUNT(*) AS traducciones_sin_producto
FROM olist.bronze.category_translation t
WHERE NOT EXISTS (
  SELECT 1
  FROM olist.bronze.products p
  WHERE p.product_category_name = t.product_category_name
);

## category_translation (71 filas)
- 0 nulos y 0 vacíos. Nombre en portugués único y nombre en inglés único: el join no duplica productos ni categorías.
- Las 71 traducciones se usan en products.

In [0]:
-- 1a. Puntuaciones válidas vs no válidas en bronze actual
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN review_score IN ('1','2','3','4','5') THEN 1 ELSE 0 END) AS validas,
  SUM(CASE WHEN review_score NOT IN ('1','2','3','4','5')
            OR review_score IS NULL THEN 1 ELSE 0 END) AS no_validas
FROM olist.bronze.order_reviews;

In [0]:
-- 1b. Largo de order_id en orders (referencia). Espera un único largo: 32.
SELECT LENGTH(order_id) AS largo, COUNT(*) AS filas
FROM olist.bronze.orders
GROUP BY LENGTH(order_id)
ORDER BY filas DESC;

In [0]:
-- 1c. Largo de order_id en order_reviews. Espera solo largo 32, sin grupo NULL.
SELECT LENGTH(order_id) AS largo, COUNT(*) AS filas
FROM olist.bronze.order_reviews
GROUP BY LENGTH(order_id)
ORDER BY filas DESC;

In [0]:
-- 14a. ¿Los review_id repetidos apuntan al mismo pedido o a pedidos distintos?
WITH rep AS (
  SELECT review_id,
         COUNT(*) AS filas,
         COUNT(DISTINCT order_id) AS pedidos_distintos
  FROM olist.bronze.order_reviews
  GROUP BY review_id
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*) AS review_ids_repetidos,
  SUM(CASE WHEN pedidos_distintos = 1 THEN 1 ELSE 0 END) AS mismo_pedido,
  SUM(CASE WHEN pedidos_distintos > 1 THEN 1 ELSE 0 END) AS varios_pedidos
FROM rep;

In [0]:
-- 14b. Distribución de reseñas por pedido.
WITH por_pedido AS (
  SELECT order_id, COUNT(*) AS n
  FROM olist.bronze.order_reviews
  GROUP BY order_id
)
SELECT n AS resenas_por_pedido, COUNT(*) AS pedidos
FROM por_pedido
GROUP BY n
ORDER BY n;

In [0]:
-- 14c. ¿(review_id, order_id) es única? Espera 0.
SELECT COUNT(*) AS llaves_repetidas
FROM (
  SELECT review_id, order_id
  FROM olist.bronze.order_reviews
  GROUP BY review_id, order_id
  HAVING COUNT(*) > 1
);

## order_reviews (99 224 filas)
- **Defecto de ingesta detectado y corregido (PR #7, P1-03):** la lectura sin multiLine partía los comentarios con saltos de línea en filas falsas.
  - Antes: 104 162 filas, 2 444 valores distintos en review_score, 2 236 order_id nulos, 68 y 18 fechas inválidas, 4 938 reseñas sin pedido.
  - Después: 99 224 filas, 5 valores (1 a 5), 0 order_id nulos (todos de 32 caracteres), 0 fechas inválidas, 0 reseñas sin pedido.
- Lección: la prueba de conteo de P1-03 usaba el mismo lector defectuoso y no podía detectarlo. 02_bronze_ingestion ahora tiene asserts de dominio.
- Puntuaciones: 1 = 11 424, 2 = 3 151, 3 = 8 179, 4 = 19 142, 5 = 57 328.
- Comentarios opcionales: title nulo en 87 656 filas y message en 58 247; 2 títulos y 9 mensajes vacíos → NULL en silver.
- review_id NO es llave: 789